In [1]:
import sys

print("Python executable:", sys.executable)
print("Python version:", sys.version)

Python executable: d:\DS Project\Vietnam_Equity_Factor_Research\.venv\Scripts\python.exe
Python version: 3.13.14 (tags/v3.13.14:fd17997, Jun 10 2026, 13:03:48) [MSC v.1944 64 bit (AMD64)]


# Vietnam Equity Factor Research

## Project 1: Identifying Persistent Stock Return Predictors in the Vietnamese Equity Market

**Research Area:** Quantitative Equity Research / Empirical Asset Pricing

**Market:** Vietnam – HOSE

**Research Period:** January 2020 – December 2025

**Frequency:** Monthly

**Benchmark:** VN-Index

### Research Objective

The objective of this project is to investigate whether selected firm-level characteristics can predict cross-sectional stock returns in the Vietnamese equity market.

The analysis focuses on three factor families:

1. **Value:** Book-to-Market (B/M), Earnings Yield (E/P).
2. **Momentum:** 6-month momentum, 12-2 month momentum.
3. **Quality:** Return on Equity (ROE), Return on Assets (ROA).

The project evaluates factor performance through cross-sectional analysis, Information Coefficient (IC), portfolio sorting, and risk-adjusted performance metrics.

### Expected Deliverables

- Reproducible Python research pipeline.
- Factor construction and performance analysis.
- Quintile portfolio performance comparison.
- Quantitative research report.
- GitHub repository with documentation.

## Research Questions

This project addresses the following research questions:

**RQ1:** Do value, momentum, and quality characteristics predict future stock returns in the Vietnamese equity market?

**RQ2:** Which factor characteristics demonstrate the strongest cross-sectional predictive power?

**RQ3:** Do stocks with stronger factor signals generate higher subsequent returns than stocks with weaker signals?

**RQ4:** Are the observed factor premiums persistent across different market conditions?

### Research Hypotheses

- H1: Stocks with higher book-to-market ratios tend to generate higher subsequent returns.
- H2: Stocks with stronger historical momentum tend to generate higher subsequent returns.
- H3: Stocks with higher profitability tend to generate higher subsequent returns.

## Research Design

### Market Universe
- Market: Ho Chi Minh Stock Exchange (HOSE).
- Initial target: 30–50 stocks.
- Universe selection will depend on historical data availability.

### Sample Period
- Start: January 2020.
- End: December 2025.
- Observation frequency: Monthly.

### Dependent Variable
- One-month-ahead stock returns.

### Independent Variables

| Factor | Variables | Expected Direction |
|---|---|---|
| Value | Book-to-Market | Positive |
| Value | Earnings Yield | Positive |
| Momentum | 6-month momentum | Positive |
| Momentum | 12-2 month momentum | Positive |
| Quality | ROE | Positive |
| Quality | ROA | Positive |

### Empirical Methods
1. Descriptive statistics.
2. Cross-sectional correlation analysis.
3. Spearman Information Coefficient.
4. Quintile portfolio construction.
5. Q5-Q1 portfolio return spread.
6. Sharpe ratio and maximum drawdown.

### Research Limitations
- Potential survivorship bias.
- Financial statement publication-date alignment.
- Corporate actions and stock liquidity.
- Transaction costs and market frictions.

In [2]:
# Data manipulation
import pandas as pd
import numpy as np

# Statistical analysis
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

# Machine learning and preprocessing
from sklearn.preprocessing import StandardScaler

# Data visualization
import matplotlib.pyplot as plt
import seaborn as sns

# File and directory management
from pathlib import Path
import os
import sys
import warnings

# Date and time
from datetime import datetime

# Display configuration
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:,.4f}".format)

sns.set_theme(style="whitegrid")

print("All libraries imported successfully!")

All libraries imported successfully!


In [3]:
# Identify project root
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

# Main directories
CONFIG_DIR = PROJECT_ROOT / "config"
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

PRICES_DIR = RAW_DIR / "prices"
FUNDAMENTALS_DIR = RAW_DIR / "fundamentals"
MARKET_DIR = RAW_DIR / "market"

NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"
SRC_DIR = PROJECT_ROOT / "src"
TESTS_DIR = PROJECT_ROOT / "tests"

RESULTS_DIR = PROJECT_ROOT / "results"
TABLES_DIR = RESULTS_DIR / "tables"
FIGURES_DIR = RESULTS_DIR / "figures"
REPORTS_DIR = RESULTS_DIR / "reports"

print("Project root:", PROJECT_ROOT)

Project root: d:\DS Project\Vietnam_Equity_Factor_Research


In [4]:
# List of directories to create
directories = [
    CONFIG_DIR,
    PRICES_DIR,
    FUNDAMENTALS_DIR,
    MARKET_DIR,
    PROCESSED_DIR,
    NOTEBOOKS_DIR,
    SRC_DIR,
    TESTS_DIR,
    TABLES_DIR,
    FIGURES_DIR,
    REPORTS_DIR
]

# Create directories if they do not exist
for directory in directories:
    directory.mkdir(parents=True, exist_ok=True)

print("Project directory structure created successfully!")

Project directory structure created successfully!


In [5]:
for directory in directories:
    relative_path = directory.relative_to(PROJECT_ROOT)
    print(f"[OK] {relative_path}")

[OK] config
[OK] data\raw\prices
[OK] data\raw\fundamentals
[OK] data\raw\market
[OK] data\processed
[OK] notebooks
[OK] src
[OK] tests
[OK] results\tables
[OK] results\figures
[OK] results\reports


In [6]:
# Market configuration
MARKET = "HOSE"
BENCHMARK = "VNINDEX"

# Research period
START_DATE = "2020-01-01"
END_DATE = "2025-12-31"

# Frequency
DATA_FREQUENCY = "Monthly"

# Investment horizon
FORWARD_RETURN_PERIOD = 1  # One-month-ahead return

# Momentum lookback periods
MOMENTUM_PERIODS = [6, 12]

# Target stock universe
MIN_STOCKS = 30
MAX_STOCKS = 50

# Portfolio configuration
N_QUANTILES = 5
REBALANCING_FREQUENCY = "Monthly"

print("Research configuration loaded successfully.")

Research configuration loaded successfully.


In [7]:
research_config = {
    "Market": MARKET,
    "Benchmark": BENCHMARK,
    "Start Date": START_DATE,
    "End Date": END_DATE,
    "Frequency": DATA_FREQUENCY,
    "Forward Return": f"{FORWARD_RETURN_PERIOD} month",
    "Momentum Periods": MOMENTUM_PERIODS,
    "Target Universe": f"{MIN_STOCKS}-{MAX_STOCKS} stocks",
    "Portfolio Quantiles": N_QUANTILES,
    "Rebalancing": REBALANCING_FREQUENCY
}

config_df = pd.DataFrame(
    research_config.items(),
    columns=["Parameter", "Value"]
)

display(config_df)

,Parameter,Value
0,Market,HOSE
1,Benchmark,VNINDEX
2,Start Date,2020-01-01
3,End Date,2025-12-31
4,Frequency,Monthly
5,Forward Return,1 month
6,Momentum Periods,"[6, 12]"
7,Target Universe,30-50 stocks
8,Portfolio Quantiles,5
9,Rebalancing,Monthly


In [8]:
print("Python version:", sys.version)
print("Python executable:", sys.executable)
print("Project root:", PROJECT_ROOT)

assert ".venv" in sys.executable, (
    "Notebook is not using the project's virtual environment!"
)

print("\nEnvironment validation passed!")

Python version: 3.13.14 (tags/v3.13.14:fd17997, Jun 10 2026, 13:03:48) [MSC v.1944 64 bit (AMD64)]
Python executable: d:\DS Project\Vietnam_Equity_Factor_Research\.venv\Scripts\python.exe
Project root: d:\DS Project\Vietnam_Equity_Factor_Research

Environment validation passed!


In [9]:
from importlib.metadata import version, PackageNotFoundError

packages = [
    "pandas",
    "numpy",
    "scipy",
    "statsmodels",
    "scikit-learn",
    "matplotlib",
    "seaborn",
    "jupyterlab",
    "duckdb",
    "pyarrow"
]

package_info = []

for package in packages:
    try:
        package_version = version(package)
    except PackageNotFoundError:
        package_version = "Not installed"

    package_info.append({
        "Package": package,
        "Version": package_version
    })

package_df = pd.DataFrame(package_info)

display(package_df)

,Package,Version
0,pandas,3.1.1
1,numpy,2.5.4
2,scipy,1.18.1
3,statsmodels,0.15.0
4,scikit-learn,1.9.1
5,matplotlib,3.11.2
6,seaborn,0.13.2
7,jupyterlab,4.6.6
8,duckdb,2.0.0
9,pyarrow,26.0.1


In [10]:
data_directories = {
    "Stock prices": PRICES_DIR,
    "Fundamentals": FUNDAMENTALS_DIR,
    "Market data": MARKET_DIR,
    "Processed data": PROCESSED_DIR
}

data_status = []

for name, directory in data_directories.items():
    files = list(directory.glob("*"))

    data_status.append({
        "Data Category": name,
        "Directory Exists": directory.exists(),
        "Number of Files": len(files),
        "Status": "Ready" if directory.exists() else "Missing"
    })

data_status_df = pd.DataFrame(data_status)

display(data_status_df)

,Data Category,Directory Exists,Number of Files,Status
0,Stock prices,True,0,Ready
1,Fundamentals,True,0,Ready
2,Market data,True,0,Ready
3,Processed data,True,0,Ready


## Initial Data Requirements

### 1. Stock Price Data

Required fields:
- Ticker
- Trading date
- Adjusted closing price
- Trading volume

### 2. Financial Statement Data

Required fields:
- Ticker
- Reporting period
- Publication date
- Net income
- Total assets
- Total equity
- Earnings per share (EPS)
- Book equity

### 3. Market Data

Required fields:
- Trading date
- VN-Index level

### Data Quality Requirements

- No look-ahead bias.
- Financial information must be aligned with actual publication dates.
- Stock returns must account for corporate actions where possible.
- Missing values must be identified and documented.
- Data sources and collection dates must be recorded.

In [11]:
checklist = {
    "Python environment configured": True,
    "Core libraries installed": True,
    "Jupyter kernel configured": True,
    "Project directories created": all(
        directory.exists() for directory in directories
    ),
    "Research parameters defined": True,
    "Stock price data collected": False,
    "Fundamental data collected": False,
    "Data quality validated": False
}

checklist_df = pd.DataFrame(
    checklist.items(),
    columns=["Task", "Completed"]
)

display(checklist_df)

completion_rate = (
    checklist_df["Completed"].sum()
    / len(checklist_df)
) * 100

print(f"Project setup completion: {completion_rate:.1f}%")

,Task,Completed
0,Python environment configured,True
1,Core libraries installed,True
2,Jupyter kernel configured,True
3,Project directories created,True
4,Research parameters defined,True
5,Stock price data collected,False
6,Fundamental data collected,False
7,Data quality validated,False


Project setup completion: 62.5%


## Conclusion

The initial research environment has been successfully established.

The project now has:
- A dedicated Python virtual environment.
- A reproducible package environment.
- A structured directory system.
- Defined research objectives and hypotheses.
- A preliminary data requirements framework.

### Next Steps

1. Identify reliable sources of Vietnamese stock market data.
2. Collect historical HOSE stock prices.
3. Collect financial statement data.
4. Construct a clean monthly stock-level panel dataset.
5. Begin factor construction and empirical analysis.

**Next Notebook:** 02_Data_Collection_and_Exploration.ipynb